[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/13_gpt2_block.ipynb)

# 🔴 Hard: GPT-2 Transformer Block

Реализуйте полный **GPT-2-style Transformer block**. Блок сохраняет residual stream формы `(B,S,D)`, сначала смешивает информацию между токенами causal attention, затем независимо преобразует признаки каждого токена через MLP.

## Карта блока

GPT-2 использует pre-norm: LayerNorm стоит перед каждой обучаемой ветвью. Residual connection обходит ветвь и складывает её результат с прежним `x`. Это помогает градиенту проходить через глубокую сеть и требует, чтобы каждая ветвь возвращала ту же форму `(B,S,D)`.

В attention-ветви нормализованный `x` проецируется в Q/K/V, делится на головы `(B,H,S,d_h)`, проходит causal scaled dot-product attention, собирается назад в `(B,S,D)` и проецируется `W_o`. В MLP-ветви каждый токен независимо проходит расширение `D → 4D`, GELU и сжатие `4D → D`.

### Пример форм

Для `x=(2,16,128)` и `num_heads=8` размер головы `d_h=16`. Scores имеют форму `(2,8,16,16)`, после объединения голов attention снова имеет `(2,16,128)`, а скрытый MLP — `(2,16,512)`. Ни residual connection, ни итог блока форму не меняют.

### Architecture (Pre-Norm)
```
x = x + causal_self_attention(ln1(x))
x = x + mlp(ln2(x))
```

### Signature
```python
class GPT2Block(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### Requirements
- Inherit from `nn.Module`
- `self.ln1`, `self.ln2`: `nn.LayerNorm(d_model)`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` for attention
- `self.mlp`: `nn.Sequential(Linear(d, 4d), GELU(), Linear(4d, d))`
- Attention must be **causal** (mask future positions)
- Pre-norm architecture (LayerNorm *before* attention and MLP)
- Residual connections around both attention and MLP

## План реализации

1. В `__init__` проверьте делимость `d_model` на `num_heads` и создайте перечисленные модули.
2. В attention helper сохраните batch/sequence, спроецируйте Q/K/V и аккуратно переставьте оси.
3. Наложите upper-triangular causal mask до softmax, объедините головы и примените `W_o`.
4. В `forward` буквально сохраните порядок pre-norm и две отдельные residual суммы.

## Быстрые проверки

- Output shape и dtype совпадают с input.
- Изменение будущего токена не меняет более ранние outputs.
- При обнулённых весах attention и MLP блок ведёт себя как identity.
- Все параметры участвуют в backward.

## Частые ошибки

- Реализовать post-norm (`ln(x + branch(x))`) вместо требуемого pre-norm.
- Обновить `x` после attention, но подать в MLP старое значение.
- Забыть scale `sqrt(d_h)` или causal mask.
- Объединить головы без возврата осей из `(B,H,S,d_h)` в `(B,S,H,d_h)`.

## Материалы для глубокого изучения

- [OpenAI GPT-2 repository](https://github.com/openai/gpt-2) — исходная модель и код.
- [Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) — технический отчёт GPT-2.
- [PyTorch Transformer building blocks](https://docs.pytorch.org/tutorials/intermediate/transformer_building_blocks.html) — современный разбор составляющих Transformer.

## Вопросы для самопроверки

1. Где именно находятся два LayerNorm в pre-norm блоке?
2. Почему MLP не смешивает разные позиции последовательности?
3. Как residual connection ограничивает допустимую форму выхода ветви?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class GPT2Block(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        pass  # Initialize layers

    def forward(self, x):
        pass  # Pre-norm + causal attention + MLP with residuals

In [ ]:
# 🧪 Debug
torch.manual_seed(0)
block = GPT2Block(d_model=64, num_heads=4)
x = torch.randn(2, 8, 64)
out = block(x)
print("Output shape:", out.shape)           # (2, 8, 64)
print("Is nn.Module?", isinstance(block, nn.Module))
print("Params:", sum(p.numel() for p in block.parameters()))

In [ ]:
from torch_judge import check
check('gpt2_block')